In [ ]:
import logging
import math
import os
import torch
from typing import Optional, List, Literal

# Third-party imports
from datasets import Dataset, load_dataset
from peft import LoraConfig as PeftLoraConfig, get_peft_model, prepare_model_for_kbit_training
from pydantic import BaseModel, Field
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer
# For vLLM, you may need to install it separately: pip install vllm
# from vllm import LLM, SamplingParams

# --- Basic Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# SECTION 1: HIERARCHICAL CONFIGURATION (Pydantic Models)
# --------------------------------------------------------------------------
# We keep the clean, hierarchical configuration.

class PeftConfig(BaseModel):
    """Configuration for Parameter-Efficient Fine-Tuning (PEFT), specifically LoRA."""
    enabled: bool = False
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = Field(
        default_factory=lambda: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    )

class QuantizationConfig(BaseModel):
    """Configuration for model quantization. '4bit' enables QLoRA."""
    mode: Optional[Literal["4bit", "8bit"]] = None

class ModelConfig(BaseModel):
    """Top-level configuration for the model."""
    id: str = "allenai/OLMo-1.7-7B-hf"
    torch_dtype: str = "auto"
    attn_implementation: Optional[Literal["flash_attention_2"]] = "flash_attention_2"
    peft: PeftConfig = Field(default_factory=PeftConfig)
    quantization: QuantizationConfig = Field(default_factory=QuantizationConfig)

class TrainingConfig(BaseModel):
    """Configuration for the training process, aligned with HF TrainingArguments."""
    output_dir: str = "./results"
    context_length: int = 1024
    learning_rate: float = 2e-5
    num_train_epochs: int = 1
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 4 # Default, will be overridden by dynamic calculation
    optim: str = "paged_adamw_8bit"
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    lr_scheduler_type: str = "cosine"
    logging_steps: int = 10
    save_strategy: str = "no" # We'll save manually

class InferenceConfig(BaseModel):
    """Configuration for the inference process."""
    max_new_tokens: int = 256
    temperature: float = 0.1
    top_p: float = 0.95


# --------------------------------------------------------------------------
# SECTION 2: CORE LLM OPERATIONS
# --------------------------------------------------------------------------

def load_model_for_training(config: ModelConfig):
    """
    Loads a model and tokenizer for training, applying quantization and PEFT.
    **ENHANCED** with robust QLoRA setup from open-instruct.
    """
    log.info(f"Loading model '{config.id}' for training...")

    # Determine torch dtype
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    # **IMPROVEMENT**: Robust quantization config inspired by open-instruct
    quant_config = None
    if config.quantization.mode == "4bit":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype, # Use bfloat16 for compute
            bnb_4bit_use_double_quant=True,
        )
    elif config.quantization.mode == "8bit":
        quant_config = BitsAndBytesConfig(load_in_8bit=True)

    model = AutoModelForCausalLM.from_pretrained(
        config.id,
        trust_remote_code=True,
        torch_dtype=dtype,
        quantization_config=quant_config,
        device_map="auto",
        attn_implementation=config.attn_implementation if torch.cuda.is_available() else None,
    )
    tokenizer = AutoTokenizer.from_pretrained(config.id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

    # **IMPROVEMENT**: Crucial step for preparing a quantized model for PEFT training.
    if config.quantization.mode:
        model = prepare_model_for_kbit_training(model)

    if config.peft.enabled:
        log.info("Applying PEFT (LoRA)...")
        peft_config = PeftLoraConfig(
            r=config.peft.lora_r,
            lora_alpha=config.peft.lora_alpha,
            lora_dropout=config.peft.lora_dropout,
            target_modules=config.peft.target_modules,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, peft_config)
        log.info("LoRA applied. Trainable parameters:")
        model.print_trainable_parameters()

    log.info("Model and tokenizer loaded successfully.")
    return model, tokenizer

# **IMPROVEMENT**: Custom trainer to use 'sum' loss, a best practice for chat models.
class SumLossSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Computes loss by summing over the sequence dimension, which weights all
        tokens equally. This can improve performance on instruction-following tasks.
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs, use_cache=False)
        logits = outputs.get("logits")

        # Shift so that tokens < n predict n
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")
        loss = loss_fct(shift_logits.view(-1, self.model.config.vocab_size), shift_labels.view(-1))

        # Normalize by the number of examples and gradient accumulation steps
        loss = loss / self.args.per_device_train_batch_size / self.args.gradient_accumulation_steps

        return (loss, outputs) if return_outputs else loss

def fine_tune_on_text(
    model, tokenizer, text_content: str, train_cfg: TrainingConfig, *, tag: str = "finetune"
):
    """
    Fine-tunes a model on a given string of text.
    **ENHANCED** to use the SumLossSFTTrainer and standardized TrainingArguments.
    """
    if not text_content or not text_content.strip():
        log.warning(f"[{tag}] Text content is empty. Skipping fine-tuning.")
        return

    log.info(f"Starting SFT for '{tag}'...")
    dataset = Dataset.from_dict({"text": [text_content]})

    # Dynamic gradient accumulation: ensures one optimizer step per text blob
    tokens = tokenizer(text_content, add_special_tokens=False, truncation=False)["input_ids"]
    num_chunks = math.ceil(len(tokens) / train_cfg.context_length) if tokens else 1
    grad_accum_steps = max(1, num_chunks)
    log.info(f"[{tag}] Tokens: {len(tokens)}, Context: {train_cfg.context_length} -> Dynamic Grad Accum Steps: {grad_accum_steps}")

    # Use the standardized TrainingArguments
    training_args = TrainingArguments(
        output_dir=os.path.join(train_cfg.output_dir, tag),
        per_device_train_batch_size=train_cfg.per_device_train_batch_size,
        gradient_accumulation_steps=grad_accum_steps, # Use our dynamic value
        learning_rate=train_cfg.learning_rate,
        num_train_epochs=train_cfg.num_train_epochs,
        optim=train_cfg.optim,
        weight_decay=train_cfg.weight_decay,
        warmup_ratio=train_cfg.warmup_ratio,
        lr_scheduler_type=train_cfg.lr_scheduler_type,
        logging_steps=train_cfg.logging_steps,
        save_strategy=train_cfg.save_strategy,
        report_to="none",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
    )

    trainer = SumLossSFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=train_cfg.context_length,
        packing=True,
        args=training_args,
    )
    trainer.train()
    log.info(f"SFT complete for '{tag}'.")

@torch.inference_mode()
def generate_text(model, tokenizer, prompt: str, config: InferenceConfig) -> str:
    """Simple inference function using Hugging Face transformers.generate."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=config.max_new_tokens,
        temperature=max(config.temperature, 1e-3),
        top_p=config.top_p,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def save_model(model, tokenizer, save_path: str):
    """
    Saves the model and tokenizer. If LoRA was used, it merges the adapters
    into the base model for easy deployment.
    """
    os.makedirs(save_path, exist_ok=True)
    if hasattr(model, "merge_and_unload"):
        log.info("Merging LoRA adapters and saving full model...")
        model = model.merge_and_unload()
    else:
        log.info("Saving full model...")

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    log.info(f"Model saved to {save_path}")